In [ ]:
import pandas as pd
import yaml

import tabular_data_explorer as explorer

In [ ]:
columns_file = "columns.yml"
# metadata_file = "downloaded_file.csv"
# uuid_file = "downloaded_uuid.csv"

metadata_file = "../input_data/Metadata_downloaded20250624_renamed.csv"
uuid_file = "../input_data/MellonUUID_downloaded20250623_renamed.csv"

In [ ]:
column_names = []

with open(columns_file) as stream:
    try:
        column_data = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

column_names = column_data['columns']
print(column_names)
print(len(column_names))

## Convert xlsx to Dataframe and extract the Papiri table

In [ ]:
#df = pd.read_excel("meta.xlsx", sheet_name="Papiri")
df = pd.read_csv(metadata_file)

In [ ]:
## Confirm
df.head(7)

## Remove the first X lines by extracting only the data rows (5 ~ end) and add Column names

In [ ]:
data_df = df.iloc[6:]
data_df.columns = column_names
data_df.shape

In [ ]:
data_df.head(3)

## Find rows and columns with no values whatsoever

In [ ]:
# Get rid of white space only cells (otherwise they are treated as holding values)
def replace_spaces_only(cell):
    if isinstance(cell, str) and cell.strip() == '':
        return None  # Replace with None, can be replaced with any value
    return cell

In [ ]:
data_df = data_df.map(replace_spaces_only)

In [ ]:
# Empty row indices 
nan_rows = data_df.isna().all(axis=1)
nan_row_indices = nan_rows[nan_rows].index
print(nan_row_indices)

In [ ]:
# Drop those rows
data_df = data_df.dropna(how='all')
data_df.shape

In [ ]:
# Confirm
nan_rows = data_df.isna().all(axis=1)
nan_row_indices = nan_rows[nan_rows].index
print(nan_row_indices)

In [ ]:
# Find empty column
drop_list = data_df.columns[data_df.isnull().all(0)].to_list()
print(drop_list)

## Clean up data

In [ ]:
# Convert NaN to None
data_df = data_df.where(pd.notnull(data_df), None)

In [ ]:
# reset index so that the first data row will be 0
data_df.reset_index(drop=True, inplace=True)
data_df

In [ ]:
# PapyrusNum forward fill 
data_df.loc[:, 'PapyrusNum'] = data_df.loc[:, 'PapyrusNum'].ffill()
data_df

In [ ]:
# If a row contains pezzo, fill in the CorniceNum (check with payrologists)

# for i in range(data_df.shape[0]):
#     pezzo = data_df.at[i,'Pezzo']
    
#     if not pd.isna(pezzo):
#         data_df.at[i, 'CorniceNum'] = data_df.at[i-1,'CorniceNum']

# data_df    

## UUID_assignment sheet

In [ ]:
uuid_df = pd.read_csv(uuid_file)

# Convert NaN to None
uuid_df = uuid_df.where(pd.notnull(uuid_df), None)

In [ ]:
result = explorer.compare_uuid_assignment(data_df, uuid_df)
print(result)

In [ ]:
max_val = 0
for r in result:
    if len(r) > max_val:
        print(r)
        max_val = len(r)
print(max_val)

In [ ]:
result_df = pd.DataFrame(result)
result_df.to_csv('uuid_comparison.csv')

In [ ]:
result_df

In [ ]:
# Find rows where (col1, col2) != (col3, col4)
condition = (result_df[1] != result_df[3]) | (result_df[2] != result_df[4])
#condition = (result_df[1] != result_df[3])

# Filter the DataFrame based on the condition
inconsistency_df = result_df[condition]
inconsistency_df

In [ ]:
inconsistency_df.to_csv('uuid_inconsistencies.csv')

In [ ]:
df_cleaned = inconsistency_df.dropna(subset=[2, 3, 4, 5, 6], how='all')
df_cleaned

In [ ]:
df_cleaned.to_csv('cleaned_uuid_inconsistencies.csv')

In [ ]:
### All the inconsitencies are in Cornici, not Pezzi